# 02 — BERT ile Sentiment Tahmini

**Model:** `nlptown/bert-base-multilingual-uncased-sentiment` — milyonlarca yorumla ince ayarlı, 1-5 yıldız tahmini yapar. Türkçe destekler.

**Yapacaklarımız:**
1. Modeli yükle
2. Yorum verisinden örneklem al (hız için)
3. BERT ile her yorumun yıldızını tahmin et
4. Gerçek `rating` ile karşılaştır (confusion matrix + classification report)
5. Tahminleri `data/processed/bert_predictions.csv` olarak kaydet

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from tqdm.auto import tqdm

from src.models.bert import (
    SENTIMENT_CLASSES,
    STAR_TO_SENTIMENT,
    load_model,
    predict_stars,
    stars_to_3class_probs,
    stars_to_sentiment,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
tokenizer, model, device = load_model()
print('Cihaz:', device)
print('Model:', model.config.name_or_path)
print('Sınıf sayısı:', model.config.num_labels)

In [ ]:
app_store = pd.read_csv(RAW_DIR / 'app_store_reviews.csv')
google_play = pd.read_csv(RAW_DIR / 'google_play_reviews.csv')
df = pd.concat([app_store, google_play], ignore_index=True)
df['text'] = df['text'].fillna('').astype(str)
df['title'] = df['title'].fillna('').astype(str)
df['full_text'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['full_text'].str.len() > 3].copy()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('Int64')
df = df.dropna(subset=['rating']).reset_index(drop=True)
df['rating'] = df['rating'].astype(int)
df['true_sentiment'] = df['rating'].map(STAR_TO_SENTIMENT)
print(f'Toplam: {len(df):,}')
df['rating'].value_counts().sort_index()

## Dengeli örneklem

CPU üzerinde 70k yorum yavaş olur. Her yıldız sınıfından eşit örneklem alıyoruz (örn. her sınıftan 600 → toplam ~3000).

In [ ]:
PER_CLASS = 600
RANDOM_STATE = 42

sample = (
    df.groupby('rating', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), PER_CLASS), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)
print(f'Örneklem boyutu: {len(sample):,}')
sample['rating'].value_counts().sort_index()

In [ ]:
texts = sample['full_text'].tolist()
BATCH = 16

all_probs = []
for i in tqdm(range(0, len(texts), BATCH), desc='BERT tahmin'):
    batch = texts[i:i + BATCH]
    p, _ = predict_stars(batch, batch_size=BATCH)
    all_probs.append(p)
star_probs = np.vstack(all_probs)
pred_stars = star_probs.argmax(axis=1) + 1
sentiment_probs = stars_to_3class_probs(star_probs)
pred_sentiments = stars_to_sentiment(pred_stars)

sample = sample.assign(
    pred_star=pred_stars,
    pred_sentiment=pred_sentiments,
    prob_negatif=sentiment_probs[:, 0],
    prob_notr=sentiment_probs[:, 1],
    prob_pozitif=sentiment_probs[:, 2],
)
sample.head(3)

## Değerlendirme — 5 sınıf yıldız

In [ ]:
y_true = sample['rating'].values
y_pred = sample['pred_star'].values

acc = accuracy_score(y_true, y_pred)
print(f'5-sınıf accuracy: {acc:.3f}')
print()
print(classification_report(y_true, y_pred, digits=3))

cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5], ax=ax)
ax.set_xlabel('Tahmin (yıldız)')
ax.set_ylabel('Gerçek (yıldız)')
ax.set_title(f'5-sınıf confusion matrix — acc {acc:.3f}')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'bert_confusion_5class.png', bbox_inches='tight')
plt.show()

## Değerlendirme — 3 sınıf sentiment (negatif/nötr/pozitif)

Bu, hocanın asıl istediği sentiment kırılımı.

In [ ]:
y_true_s = sample['true_sentiment'].values
y_pred_s = sample['pred_sentiment'].values

acc3 = accuracy_score(y_true_s, y_pred_s)
print(f'3-sınıf accuracy: {acc3:.3f}')
print()
print(classification_report(y_true_s, y_pred_s, labels=SENTIMENT_CLASSES, digits=3))

cm3 = confusion_matrix(y_true_s, y_pred_s, labels=SENTIMENT_CLASSES)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm3, annot=True, fmt='d', cmap='Greens',
            xticklabels=SENTIMENT_CLASSES, yticklabels=SENTIMENT_CLASSES, ax=ax)
ax.set_xlabel('Tahmin')
ax.set_ylabel('Gerçek')
ax.set_title(f'3-sınıf sentiment confusion — acc {acc3:.3f}')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'bert_confusion_3class.png', bbox_inches='tight')
plt.show()

## Örnek tahminler

In [ ]:
display_cols = ['app_name', 'rating', 'pred_star', 'true_sentiment', 'pred_sentiment', 'full_text']
print('Doğru tahmin örnekleri:')
correct = sample[sample['true_sentiment'] == sample['pred_sentiment']].sample(5, random_state=1)
display(correct[display_cols])

print('Yanlış tahmin örnekleri:')
wrong = sample[sample['true_sentiment'] != sample['pred_sentiment']]
if len(wrong) > 0:
    display(wrong.sample(min(5, len(wrong)), random_state=1)[display_cols])

In [ ]:
out_path = PROCESSED_DIR / 'bert_predictions.csv'
save_cols = [
    'review_id', 'platform', 'app_name', 'rating',
    'true_sentiment', 'pred_star', 'pred_sentiment',
    'prob_negatif', 'prob_notr', 'prob_pozitif',
    'full_text',
]
sample[save_cols].to_csv(out_path, index=False)
print(f'Kaydedildi: {out_path} ({len(sample):,} satır)')

## Çıktılar

- `visuals/bert_confusion_5class.png` — 5-sınıf yıldız confusion matrix
- `visuals/bert_confusion_3class.png` — 3-sınıf sentiment confusion matrix
- `data/processed/bert_predictions.csv` — örneklem üzerinde BERT tahminleri (tüm olasılıklarla)

**Sonraki adım:** `notebooks/03_word_attributions_lime.ipynb` — modelin hangi kelimeleri pozitif/negatif sinyali olarak gördüğünü LIME ile görselleştir.